# LangChain: Chains (Groq Llama 3.1)

## Outline
* **LLMChain** - Basic chain combining LLM + Prompt
* **Sequential Chains** - Running chains one after another
  * SimpleSequentialChain - Single input/output per chain
  * SequentialChain - Multiple inputs/outputs per chain
* **Router Chain** - Route inputs to specialized chains

**Model Used:** Groq Llama-3.1-8b-instant

Chains are the fundamental building blocks in LangChain that combine LLMs with prompts to create complex workflows.


In [48]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv pandas -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setting Up the Environment

We just installed the necessary Python packages: LangChain for building AI chains, LangChain-Groq for connecting to the Groq AI service, and python-dotenv for managing API keys securely.

In [49]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


✅ Environment loaded successfully


## Loading API Credentials

We loaded the environment variables from a .env file, which securely stores our Groq API key. This key allows us to access the powerful AI models without exposing sensitive information in the code.

In [3]:
# Cell 4: Load Sample Data

# Load product reviews data
# Make sure Data.csv is in the same directory or provide the full path
df = pd.read_csv('Data.csv')

print("Dataset loaded successfully!")
print(f"Number of reviews: {len(df)}")
print("\nFirst few rows:")
df.head()


Dataset loaded successfully!
Number of reviews: 7

First few rows:


,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## Loading Sample Data

We loaded a dataset of product reviews that we'll use to demonstrate chain capabilities. This gives us realistic text data to process through our AI chains.

In [4]:
# Cell 5: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

print(f"✅ Model: {llm_model}")


✅ Model: llama-3.1-8b-instant


## Initializing the AI Model

We connected to the Groq AI service and initialized the Llama 3.1 model. This powerful AI will be the "brain" that powers all our chains, processing text and generating responses.

## LLMChain

The **LLMChain** is the most basic chain in LangChain. It combines:
- A **prompt template** (with variables)
- An **LLM** (language model)

Together they create a reusable chain that formats prompts and gets responses.


In [50]:
# Cell 7: Create Basic LLMChain

from langchain_core.prompts import ChatPromptTemplate

# Initialize LLM with high temperature for creative outputs
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Create prompt template
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# Create the chain by combining LLM + Prompt
chain = prompt | llm

print("✅ Chain created successfully")


✅ Chain created successfully


## Creating a Basic Chain

We built our first chain by combining a text prompt template with the AI model. This chain takes a product name as input and generates creative company name suggestions. It's the simplest form of an AI workflow.

In [51]:
# Cell 8: Run LLMChain with Sample Product

# Define a product
product = "Queen Size Sheet Set"

# Run the chain
response = chain.invoke({"product": product})

print(f"Product: {product}")
print(f"Suggested Company Name: {response.content}")


Product: Queen Size Sheet Set
Suggested Company Name: Here are some suggestions for company name ideas that could fit a business making Queen Size Sheet Sets:

1. **Royal Comfort Supply**: This name plays on the queen size aspect and implies a high level of comfort and quality.
2. **Soft Haven**: This name emphasizes the cozy and soft nature of the sheet sets, which is perfect for a bedding company.
3. **Bedding Royalty**: This name leans into the "queen size" aspect and adds a touch of luxury and regality.
4. **Dreamweaver Bedding**: This name evokes a sense of creativity and high-quality craftsmanship, which could appeal to customers looking for a premium product.
5. **Slumbercraft Co.**: This name emphasizes the company's focus on crafting high-quality sleep solutions.
6. **Regal Sheets**: This name is straightforward and emphasizes the royal nature of the queen size sheet sets.
7. **Snuggle Nest**: This name is cozy and inviting, which could appeal to customers looking for a comfor

## Testing the Basic Chain

The chain took "Queen Size Sheet Set" as input and generated multiple creative company name ideas. This shows how a simple chain can transform basic input into creative output.

## SimpleSequentialChain

This chain runs multiple chains in sequence where:
- Each chain has **one input** and **one output**
- The output of one chain becomes the input of the next

**Use Case:** When you need to process data through multiple steps in a pipeline.


In [10]:
# Cell 10: Create SimpleSequentialChain

from langchain_core.runnables import RunnablePassthrough

# Initialize LLM
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Chain 1: Generate company name from product
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain_one = first_prompt | llm

# Chain 2: Generate company description from company name
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company: {company_name}"
)
chain_two = second_prompt | llm

# Function to extract company name from first response
def extract_company_name(response):
    # Assume the response is the company name
    return {"company_name": response.content.split('\n')[0].strip()}

# Combine into sequential chain
overall_simple_chain = chain_one | extract_company_name | chain_two

print("✅ Sequential chain created with 2 chains")


✅ Sequential chain created with 2 chains


## Building Sequential Chains

We created a chain that runs two steps in sequence: first generate a company name from a product idea, then write a description for that company. This demonstrates how to connect multiple AI operations into a workflow.

In [52]:
# Cell 11: Run SimpleSequentialChain

# Run the chain with our product
result = overall_simple_chain.invoke({"product": product})

print("\nFinal Output:")
print(result.content)



Final Output:
"Royal Slumber Co. - Luxury Queen Size sheet sets craftsmanship, blending comfort and style for restful nights and serene mornings."


## Testing the Sequential Chain

We ran our two-step chain with a product idea to see it in action. The chain first generated a creative company name, then wrote a compelling description for that company.

## SequentialChain

Unlike SimpleSequentialChain, this advanced chain supports:
- **Multiple inputs** per chain
- **Multiple outputs** per chain
- **Named variables** to track intermediate results

**Use Case:** Complex workflows where chains need access to multiple previous outputs.


In [14]:
# Cell 13: Create SequentialChain - Chain 1 (Translation)

# Initialize LLM
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Chain 1: Translate review to English
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:\n\n{Review}"
)
chain_one = first_prompt | llm

print("✅ Chain 1: Translation chain created")


✅ Chain 1: Translation chain created


## Building Chain 1: Translation

We created the first chain in our sequential workflow - it takes a review in any language and translates it to English, preparing it for further processing.

In [16]:
# Cell 14: Create SequentialChain - Chain 2 (Summarization)

# Chain 2: Summarize the English review
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_two = second_prompt | llm

print("✅ Chain 2: Summarization chain created")


✅ Chain 2: Summarization chain created


## Building Chain 2: Summarization

We created the second chain that takes the translated English review and summarizes it into a single, concise sentence.

In [18]:
# Cell 15: Create SequentialChain - Chain 3 (Language Detection)

# Chain 3: Detect original language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = third_prompt | llm

print("✅ Chain 3: Language detection chain created")


✅ Chain 3: Language detection chain created


## Building Chain 3: Language Detection

We created a third chain that analyzes the original review and identifies which language it was written in.

In [20]:
# Cell 16: Create SequentialChain - Chain 4 (Follow-up)

# Chain 4: Generate follow-up response (uses 2 inputs!)
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = fourth_prompt | llm

print("✅ Chain 4: Follow-up response chain created")


✅ Chain 4: Follow-up response chain created


## Building Chain 4: Follow-up Response

We created the final chain that generates a follow-up response using both the summary and the detected language - demonstrating how chains can use multiple inputs from previous steps.

In [22]:
# Cell 17: Combine into SequentialChain

# Function to run the chains sequentially
def overall_chain(review):
    # Chain 1: Translate
    eng_review = chain_one.invoke({"Review": review}).content
    
    # Chain 2: Summarize
    summary = chain_two.invoke({"English_Review": eng_review}).content
    
    # Chain 3: Detect language
    language = chain_three.invoke({"Review": review}).content
    
    # Chain 4: Follow-up
    followup = chain_four.invoke({"summary": summary, "language": language}).content
    
    return {
        "English_Review": eng_review,
        "summary": summary,
        "language": language,
        "followup_message": followup
    }

print("✅ Sequential chain created with 4 sub-chains")
print("\nChain Flow:")
print("Review -> [Translate] -> English_Review -> [Summarize] -> summary")
print("Review -> [Detect Language] -> language")
print("summary + language -> [Follow-up] -> followup_message")
print("1. Review → English_Review")
print("2. English_Review → summary")
print("3. Review → language")
print("4. summary + language → followup_message")


✅ Sequential chain created with 4 sub-chains

Chain Flow:
Review -> [Translate] -> English_Review -> [Summarize] -> summary
Review -> [Detect Language] -> language
summary + language -> [Follow-up] -> followup_message
1. Review → English_Review
2. English_Review → summary
3. Review → language
4. summary + language → followup_message


## Combining Chains into Sequential Workflow

We created a custom function that orchestrates all four chains in sequence, managing the flow of data between steps and collecting all the intermediate and final results.

In [23]:
# Cell 18: Run SequentialChain on Real Review

# Get a review from the dataset (index 5)
review = df.Review[5]

print("Original Review:")
print("="*60)
print(review)
print("\n" + "="*60 + "\n")

# Run the sequential chain
result = overall_chain(review)

print("\nFinal Results:")
print("="*60)
for key, value in result.items():
    if key != 'Review':  # Skip echoing the input
        print(f"\n{key}:")
        print(value)


Original Review:
Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...
Vieux lot ou contrefaçon !?



Final Results:

English_Review:
Voici la traduction de la critique en anglais :

Je trouve le goût médiocre. La mousse n'est pas stable, c'est étrange. J'achète les mêmes dans le commerce et le goût est beaucoup mieux...
Je me demande si c'est un lot vieillissant ou un faux produit !

Nota : La critique semble être dirigée vers un produit alimentaire (probablement un dessert) qui répond aux attentes, mais ce n'est pas le cas avec le produit acheté en ligne. Le critique s'inquiète de la qualité du produit et soupçonne une faute de fabrication ou une contrefaçon.

summary:
Le critique est mécontent de son achat en ligne, car le goût du produit est médiocre, la mousse est instable et il soupçonne une faute de fabrication ou une contrefaçon.

language:
Le langage de ce review est le français.

followup_message

## Testing the Complete Sequential Chain

We ran our complex 4-step workflow on a real product review from the dataset, demonstrating how all the chains work together to translate, summarize, detect language, and generate a follow-up response.

## Router Chain

A **Router Chain** intelligently routes inputs to specialized sub-chains based on the input content.

**How it works:**
1. Input comes in
2. Router LLM decides which expert chain to use
3. Input is sent to that specialized chain
4. Response is returned

**Use Case:** Building a multi-domain assistant (e.g., one expert for physics, another for math, etc.)


In [24]:
# Cell 20: Define Subject-Specific Prompt Templates

# Physics expert prompt
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise \
and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

# Math expert prompt
math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, \
answer the component parts, and then put them together \
to answer the broader question.

Here is a question:
{input}"""

# History expert prompt
history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people, \
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence \
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""

# Computer Science expert prompt
computerscience_template = """You are a successful computer scientist. \
You have a passion for creativity, collaboration, \
forward-thinking, confidence, strong problem-solving capabilities, \
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

print("✅ Expert prompt templates defined for 4 subjects")


✅ Expert prompt templates defined for 4 subjects


## Creating Expert Prompt Templates

We defined specialized prompt templates for four different subjects (physics, math, history, and computer science), each with expert-level instructions tailored to that domain.

In [25]:
# Cell 21: Create Prompt Info Dictionary

# Define metadata for each expert chain
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "history", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

print("✅ Prompt info configured:")
for info in prompt_infos:
    print(f"  - {info['name']}: {info['description']}")


✅ Prompt info configured:
  - physics: Good for answering questions about physics
  - math: Good for answering math questions
  - history: Good for answering history questions
  - computer science: Good for answering computer science questions


## Configuring Expert Chain Metadata

We created a dictionary of information for each expert chain, including the subject name, description, and the corresponding prompt template - this metadata will help route questions to the right expert.

In [27]:
# Cell 22: Import Router Chain Components

# Note: Router chains are deprecated in LangChain 1.x
# Using LCEL for routing instead

print("✅ Router functionality will use LCEL")


✅ Router functionality will use LCEL


## Adapting to LangChain 1.x

Router chains were deprecated in LangChain 1.x, so we're using the modern LCEL (LangChain Expression Language) approach to implement routing functionality.

In [ ]:
# Cell 23: Create Destination Chains

# Initialize LLM with temperature=0 for consistent routing
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

# Create a chain for each expert domain
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    expert_chain = prompt | llm
    destination_chains[name] = expert_chain

print(f"✅ Created {len(destination_chains)} destination chains:")
for name in destination_chains.keys():
    print(f"  - {name}")


✅ Created 4 destination chains:
  - physics
  - math
  - history
  - computer science


## Creating Specialized Expert Chains

We built individual chains for each subject area (physics, math, history, computer science) by combining their specialized prompts with the AI model.

In [31]:
# Cell 24: Create Default Chain

# Default chain for questions that don't match any expert
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm

print("✅ Default chain created (for non-specialized questions)")


✅ Default chain created (for non-specialized questions)


## Creating a Fallback Chain

We created a default chain that handles questions that don't fit into any of our specialized expert categories, ensuring the system can respond to any type of question.

In [33]:
# Cell 25: Define Router Template

# Create destinations string for the router
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

# Router template - instructs the LLM how to route
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising \
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
REMEMBER: "destination" MUST be one of the candidate prompt names below OR "DEFAULT".
REMEMBER: "next_inputs" can just be the original input if no modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (must include ```json) >>"""

# Format the template with our destinations
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
destinations=destinations_str
)

print("✅ Router template created")
print("\nAvailable destinations:")
print(destinations_str)

✅ Router template created

Available destinations:
physics: Good for answering questions about physics
math: Good for answering math questions
history: Good for answering history questions
computer science: Good for answering computer science questions


## Defining the Router Logic

We created a template that instructs the AI how to analyze questions and route them to the appropriate expert chain based on the subject matter.

In [35]:
# Cell 26: Create Router Chain

# Simple router function (replacing deprecated RouterChain)
def route_question(input_text):
    input_lower = input_text.lower()
    if any(word in input_lower for word in ["physics", "force", "energy", "gravity", "quantum"]):
        return "physics"
    elif any(word in input_lower for word in ["math", "calculate", "equation", "algebra", "geometry"]):
        return "math"
    elif any(word in input_lower for word in ["history", "war", "president", "civilization", "ancient"]):
        return "history"
    elif any(word in input_lower for word in ["computer", "programming", "algorithm", "code", "software"]):
        return "computer science"
    else:
        return "DEFAULT"

router_chain = route_question

print("✅ Simple router created (keyword-based routing)")

print("✅ Router chain created")

✅ Simple router created (keyword-based routing)
✅ Router chain created


## Creating the Router Function

We implemented a simple keyword-based router that analyzes the input question and determines which expert chain should handle it, falling back to the default chain if no match is found.

In [55]:
# Cell 27: Create MultiPromptChain

# Function to simulate MultiPromptChain
def multi_prompt_chain(input_text):
    destination = router_chain(input_text)
    if destination in destination_chains:
        chain = destination_chains[destination]
        return chain.invoke({"input": input_text}).content
    else:
        return default_chain.invoke({"input": input_text}).content

chain = multi_prompt_chain

print("✅ MultiPromptChain created successfully")
print("\nThis chain will:")
print("- Route questions to appropriate expert chains")
print("- Fall back to default chain for unrecognized topics")
print("1. Analyze the input question")
print("2. Route to the appropriate expert (physics/math/history/cs)")
print("3. Return the expert's response")


✅ MultiPromptChain created successfully

This chain will:
- Route questions to appropriate expert chains
- Fall back to default chain for unrecognized topics
1. Analyze the input question
2. Route to the appropriate expert (physics/math/history/cs)
3. Return the expert's response


## Creating the Multi-Expert Chain

We built a master chain that combines the router with all the expert chains, automatically directing questions to the right specialist based on the topic.

In [56]:
# Cell 28: Test Router Chain - Physics Question

# Ask a physics question
response = chain("What is black body radiation?")

print("\nResponse:")
print(response)



Response:
Black-body radiation is the thermal electromagnetic radiation emitted by an object in thermal equilibrium, meaning it has a uniform temperature and is in contact with its surroundings. This type of radiation is a fundamental concept in physics and is a key aspect of the theory of thermal radiation.

When an object is heated, it starts to emit electromagnetic radiation across a wide range of wavelengths, including visible light, ultraviolet (UV) radiation, infrared (IR) radiation, and even X-rays. The radiation is emitted in all directions and is a result of the thermal motion of the object's particles.

The key characteristics of black-body radiation are:

1. **Black-body**: The object emits radiation as if it were a perfect absorber of radiation, meaning it absorbs all incident radiation without reflecting any of it. This is why it's called a "black" body.
2. **Thermal equilibrium**: The object is in thermal equilibrium with its surroundings, meaning its temperature is unif

## Testing the Router: Physics Question

We tested the routing system with a physics question to see if it correctly routes to the physics expert and gets an appropriate response.

In [57]:
# Cell 29: Test Router Chain - Math Question

# Ask a math question
response = chain("What is 2 + 2?")

print("\nResponse:")
print(response)



Response:
2 + 2 = 4.


## Testing the Router: Math Question

We tested with a simple math question to verify the router correctly identifies math-related queries and routes them to the math expert.

In [43]:
# Cell 30: Test Router Chain - Non-Expert Question

# Ask a question outside the expert domains (e.g., biology)
response = chain("Why does DNA replicate?")

print("\nResponse:")
print(response)
print("\n⚠️ This should route to DEFAULT chain since biology isn't in our expert domains")



Response:
DNA replication is a fundamental process in living organisms that serves several essential purposes. The primary reasons for DNA replication are:

1. **Cell Division**: DNA replication is necessary for cell division, which is the process by which a cell divides into two daughter cells. Each daughter cell receives a complete set of genetic instructions, ensuring that the new cells have the same genetic material as the parent cell.

2. **Genetic Information Preservation**: DNA replication ensures that the genetic information is preserved and passed on from one generation to the next. This is crucial for the continuation of life and the transmission of traits from parents to offspring.

3. **Repair and Maintenance**: DNA replication also allows for the repair of damaged DNA. During replication, any errors or damage to the DNA molecule can be corrected, ensuring that the genetic information remains accurate and intact.

4. **Evolution and Adaptation**: DNA replication enables ge

## Testing the Router: Non-Expert Question

We tested with a biology question that doesn't match any of our expert domains to verify it correctly falls back to the default chain.

In [45]:
# Cell 31: Test Router Chain - History Question

# Ask a history question
response = chain("Who was the first president of the United States?")

print("\nResponse:")
print(response)



Response:
The first president of the United States was George Washington. He was inaugurated on April 30, 1789, and served two terms in office until March 4, 1797. Washington was a key figure in the American Revolutionary War, serving as the commander-in-chief of the Continental Army, and he played a crucial role in the drafting and ratification of the United States Constitution.

Washington's leadership and integrity earned him the respect and admiration of his contemporaries, and he was unanimously elected as the first president of the United States by the Electoral College. He set important precedents for the office of the presidency, including the decision to serve only two terms and the establishment of a cabinet system.

Washington's presidency was marked by significant challenges, including the Whiskey Rebellion, a tax protest in western Pennsylvania that he put down with military force. He also established the United States' first cabinet, which included the Secretary of State

In [47]:
# Cell 32: Test Router Chain - Computer Science Question

# Ask a computer science question
response = chain("What is the time complexity of binary search?")

print("\nResponse:")
print(response)



Response:
The time complexity of binary search is O(log n), where n is the number of elements in the array or list being searched.

Binary search works by repeatedly dividing the search interval in half and searching for the target value in one of the two halves. This process continues until the target value is found or the search interval is empty.

The time complexity of O(log n) is because with each iteration, the search space is reduced by half. This results in a logarithmic number of iterations, which is why binary search is much faster than linear search (which has a time complexity of O(n)) for large datasets.

To be more precise, the time complexity of binary search can be expressed as:

T(n) = O(log2 n) = O(log n)

where T(n) is the time complexity and n is the number of elements in the array or list being searched.


## Testing the Router: History Question

We tested with a history question to confirm the router correctly identifies historical queries and routes them to the history expert.

## Summary: Chain Types in LangChain

### 1. **LLMChain**
- **Simplest chain**: LLM + Prompt Template
- Use when you need a single-step prompt with variables

### 2. **SimpleSequentialChain**
- **Pipeline of chains**: Output of chain N → Input of chain N+1
- Limitation: Only 1 input and 1 output per chain
- Use for linear workflows (e.g., translate → summarize)

### 3. **SequentialChain**
- **Advanced pipeline**: Multiple inputs/outputs per chain
- Named variables track intermediate results
- Use for complex workflows where chains need multiple previous outputs

### 4. **Router Chain**
- **Intelligent routing**: Directs questions to specialized expert chains
- Uses LLM to decide which sub-chain to invoke
- Use for multi-domain assistants or specialized knowledge bases

---

## Key Concepts

| Concept | Description |
|---------|-------------|
| **Chain** | Combines LLM + Prompt + Logic into reusable component |
| **Sequential Flow** | Chains execute one after another |
| **Named Variables** | Track intermediate outputs with `output_key` |
| **Routing** | LLM decides which expert chain to use |
| **Default Chain** | Fallback when no expert matches |

---